In [63]:
import json
import base64
import qrcode
import requests
import datetime
import uuid
import matplotlib.pyplot as plt
from pymongo import MongoClient
from typing import Optional, List
from didcomm.common.types import DID, VerificationMethodType, VerificationMaterial, VerificationMaterialFormat
from didcomm.did_doc.did_doc import DIDDoc, VerificationMethod, DIDCommService
from didcomm.did_doc.did_resolver import DIDResolver
from didcomm.message import Message, FromPrior
from didcomm.secrets.secrets_resolver_demo import SecretsResolverDemo
from didcomm.unpack import unpack, UnpackResult
from didcomm.message import Attachment, AttachmentDataJson, AttachmentDataLinks
from didcomm.common.resolvers import ResolversConfig
from didcomm.pack_encrypted import pack_encrypted, PackEncryptedConfig, PackEncryptedResult
from peerdid.core.did_doc_types import DIDCommServicePeerDID
from didcomm.secrets.secrets_util import generate_x25519_keys_as_jwk_dict, generate_ed25519_keys_as_jwk_dict, jwk_to_secret
from peerdid import peer_did
from peerdid.did_doc import DIDDocPeerDID
from peerdid.types import VerificationMaterialAuthentication, VerificationMethodTypeAuthentication, VerificationMaterialAgreement, VerificationMethodTypeAgreement, VerificationMaterialFormatPeerDID

In [64]:
secrets_resolver = SecretsResolverDemo()

In [65]:
class DIDResolverPeerDID(DIDResolver):
    async def resolve(self, did: DID) -> DIDDoc:
        did_doc_json = peer_did.resolve_peer_did(did, format = VerificationMaterialFormatPeerDID.JWK)
        did_doc = DIDDocPeerDID.from_json(did_doc_json)

        return DIDDoc(
            did=did_doc.did,
            key_agreement_kids = did_doc.agreement_kids,
            authentication_kids = did_doc.auth_kids,
            verification_methods = [
                VerificationMethod(
                    id = m.id,
                    type = VerificationMethodType.JSON_WEB_KEY_2020,
                    controller = m.controller,
                    verification_material = VerificationMaterial(
                        format = VerificationMaterialFormat.JWK,
                        value = json.dumps(m.ver_material.value)
                    )
                )
                for m in did_doc.authentication + did_doc.key_agreement
            ],
             didcomm_services = []
#                 DIDCommService(
#                     id = s.id,
#                     service_endpoint = s.service_endpoint[0]["uri"] if "uri" in s.service_endpoint[0] else s.service_endpoint[0],
#                     routing_keys = s.routing_keys,
#                     accept = s.accept
#                 )
#                 for s in did_doc.service
#                 if isinstance(s, DIDCommServicePeerDID)
#             ] if did_doc.service else []
        )

In [66]:
async def create_peer_did(auth_keys_count: int = 1,
                        agreement_keys_count: int = 1,
                        service_endpoint: Optional[str] = None,
                        service_routing_keys: Optional[List[str]] = None
                        ) -> str:
        # 1. generate keys in JWK format
        agreem_keys = [generate_x25519_keys_as_jwk_dict() for _ in range(agreement_keys_count)]
        auth_keys = [generate_ed25519_keys_as_jwk_dict() for _ in range(auth_keys_count)]

        # 2. prepare the keys for peer DID lib
        agreem_keys_peer_did = [
            VerificationMaterialAgreement(
                type=VerificationMethodTypeAgreement.JSON_WEB_KEY_2020,
                format=VerificationMaterialFormatPeerDID.JWK,
                value=k[1],
            )
            for k in agreem_keys
        ]
        auth_keys_peer_did = [
            VerificationMaterialAuthentication(
                type=VerificationMethodTypeAuthentication.JSON_WEB_KEY_2020,
                format=VerificationMaterialFormatPeerDID.JWK,
                value=k[1],
            )
            for k in auth_keys
        ]
        data = {
            "id": "#didcomm",
            "type": "DIDCommMessaging",
            "serviceEndpoint": {
                "accept": ["didcomm/v2"],
                "routingKeys": [],
                "uri": "http://localhost:8080"
            }
        }

        # 3. generate service
        service = None
        if service_endpoint:
            service = json.dumps(data)

        # 4. call peer DID lib
        # if we have just one key (auth), then use numalg0 algorithm
        # otherwise use numalg2 algorithm
        if len(auth_keys_peer_did) == 1 and not agreem_keys_peer_did and not service:
            did = peer_did.create_peer_did_numalgo_0(auth_keys_peer_did[0])
        else:
            did = peer_did.create_peer_did_numalgo_2(
                encryption_keys=agreem_keys_peer_did,
                signing_keys=auth_keys_peer_did,
                service=service,
            )

        # 5. set KIDs as in DID DOC for secrets and store the secret in the secrets resolver
        did_doc = DIDDocPeerDID.from_json(peer_did.resolve_peer_did(did))
        for auth_key, kid in zip(auth_keys, did_doc.auth_kids):
            private_key = auth_key[0]
            private_key["kid"] = kid
            await secrets_resolver.add_key(jwk_to_secret(private_key))

        for agreem_key, kid in zip(agreem_keys, did_doc.agreement_kids):
            private_key = agreem_key[0]
            private_key["kid"] = kid
            await secrets_resolver.add_key(jwk_to_secret(private_key))

        return did

### Copy DID from Alice notebook (Out Of Band Invitation)

In [67]:
alice_did = "did:peer:2.Ez6LSbyMVq9Rivvdt3S8cij1UXSf2REhHPL5ARrEPziht1E7U.Vz6MkqbZnxHtegqD6tFwjprvrWi1sRU62zymUHMND8ysQzDkk.SeyJpZCI6IiNkaWRjb21tIiwidCI6ImRtIiwicyI6eyJhIjpbImRpZGNvbW0vdjIiXSwiciI6W10sInVyaSI6Imh0dHA6Ly9sb2NhbGhvc3Q6ODA4MCJ9fQ"

### Get mediator DID from Alice DID and endpoint from mediator DID

In [55]:
alice_did_doc = json.loads(peer_did.resolve_peer_did(alice_did))
print(alice_did_doc)
mediator_did = alice_did_doc["service"][0]["serviceEndpoint"]["uri"]
#mediator_did = alice_did_doc["service"][0]["serviceEndpoint"][]
print(mediator_did)


{'id': 'did:peer:2.Ez6LSbyMVq9Rivvdt3S8cij1UXSf2REhHPL5ARrEPziht1E7U.Vz6MkqbZnxHtegqD6tFwjprvrWi1sRU62zymUHMND8ysQzDkk.SeyJpZCI6IiNkaWRjb21tIiwidCI6ImRtIiwicyI6eyJhIjpbImRpZGNvbW0vdjIiXSwiciI6W10sInVyaSI6Imh0dHA6Ly9sb2NhbGhvc3Q6ODA4MCJ9fQ', 'authentication': [{'id': 'did:peer:2.Ez6LSbyMVq9Rivvdt3S8cij1UXSf2REhHPL5ARrEPziht1E7U.Vz6MkqbZnxHtegqD6tFwjprvrWi1sRU62zymUHMND8ysQzDkk.SeyJpZCI6IiNkaWRjb21tIiwidCI6ImRtIiwicyI6eyJhIjpbImRpZGNvbW0vdjIiXSwiciI6W10sInVyaSI6Imh0dHA6Ly9sb2NhbGhvc3Q6ODA4MCJ9fQ#6MkqbZnxHtegqD6tFwjprvrWi1sRU62zymUHMND8ysQzDkk', 'type': 'Ed25519VerificationKey2020', 'controller': 'did:peer:2.Ez6LSbyMVq9Rivvdt3S8cij1UXSf2REhHPL5ARrEPziht1E7U.Vz6MkqbZnxHtegqD6tFwjprvrWi1sRU62zymUHMND8ysQzDkk.SeyJpZCI6IiNkaWRjb21tIiwidCI6ImRtIiwicyI6eyJhIjpbImRpZGNvbW0vdjIiXSwiciI6W10sInVyaSI6Imh0dHA6Ly9sb2NhbGhvc3Q6ODA4MCJ9fQ', 'publicKeyMultibase': 'z6MkqbZnxHtegqD6tFwjprvrWi1sRU62zymUHMND8ysQzDkk'}], 'keyAgreement': [{'id': 'did:peer:2.Ez6LSbyMVq9Rivvdt3S8cij1UXSf2REhHPL5ARrEPziht1E7U.Vz6

In [56]:
mediator_did_doc = json.loads(peer_did.resolve_peer_did(mediator_did))
mediator_endpoint = mediator_did_doc["service"]["serviceEndpoint"]["uri"]
print(mediator_endpoint)

MalformedPeerDIDError: Invalid peer DID provided. Does not match peer DID regexp.

### BOB creates DID and a basic msg to Bob

In [49]:
bob_did_to_alice = await create_peer_did(1,1, service_endpoint="https://www.example.com/bob" )
print("Bob's DID:", bob_did_to_alice)

Bob's DID: did:peer:2.Ez6LSmwARVMqRyHEEAfMoBGqMn8d5HJ3fnCk4VHvX1DAsdWCp.Vz6Mkenjjpz3GBL3mXmCXDogzFT25XeY3Qu6z3EqZAtLrcKY4.SeyJpZCI6IiNkaWRjb21tIiwidCI6ImRtIiwicyI6eyJhIjpbImRpZGNvbW0vdjIiXSwiciI6W10sInVyaSI6Imh0dHA6Ly9sb2NhbGhvc3Q6ODA4MCJ9fQ


In [50]:

bob_basic_message = Message(
    id = str(uuid.uuid4()),
    type="https://didcomm.org/basicmessage/2.0/message",
    body={"content": "Argentina or Croacia at World Cup finals?"},
    created_time= int(datetime.datetime.now().timestamp())            
)

In [51]:
bob_basic_message_packed = await pack_encrypted(
    resolvers_config = ResolversConfig(
        secrets_resolver = secrets_resolver,
        did_resolver = DIDResolverPeerDID()
    ),
    message = bob_basic_message,
    frm = bob_did_to_alice,
    to = alice_did,
    sign_frm = None,
    pack_config = PackEncryptedConfig(protect_sender_id=False)
)
print(bob_basic_message_packed)

PackEncryptedResult(packed_msg='{"protected":"eyJ0eXAiOiJhcHBsaWNhdGlvbi9kaWRjb21tLWVuY3J5cHRlZCtqc29uIiwiYWxnIjoiRUNESC0xUFUrQTI1NktXIiwiZW5jIjoiQTI1NkNCQy1IUzUxMiIsImFwdSI6IlpHbGtPbkJsWlhJNk1pNUZlalpNVTIxM1FWSldUWEZTZVVoRlJVRm1UVzlDUjNGTmJqaGtOVWhLTTJadVEyczBWa2gyV0RGRVFYTmtWME53TGxaNk5rMXJaVzVxYW5CNk0wZENURE50V0cxRFdFUnZaM3BHVkRJMVdHVlpNMUYxTm5velJYRmFRWFJNY21OTFdUUXVVMlY1U25CYVEwazJTV2xPYTJGWFVtcGlNakYwU1dsM2FXUkRTVFpKYlZKMFNXbDNhV041U1RabGVVcG9TV3B3WWtsdFVuQmFSMDUyWWxjd2RtUnFTV2xZVTNkcFkybEpObGN4TUhOSmJsWjVZVk5KTmtsdGFEQmtTRUUyVEhrNWMySXlUbWhpUjJoMll6TlJOazlFUVRSTlEwbzVabEVqTmt4VGJYZEJVbFpOY1ZKNVNFVkZRV1pOYjBKSGNVMXVPR1ExU0VvelptNURhelJXU0haWU1VUkJjMlJYUTNBIiwiYXB2IjoiYXhfZ2J3ZjFIRkgyX2NOckRtMHYtU2JZUkZjSTdQcDE4RFFDTTFaVXJmZyIsInNraWQiOiJkaWQ6cGVlcjoyLkV6NkxTbXdBUlZNcVJ5SEVFQWZNb0JHcU1uOGQ1SEozZm5DazRWSHZYMURBc2RXQ3AuVno2TWtlbmpqcHozR0JMM21YbUNYRG9nekZUMjVYZVkzUXU2ejNFcVpBdExyY0tZNC5TZXlKcFpDSTZJaU5rYVdSamIyMXRJaXdpZENJNkltUnRJaXdpY3lJNmV5SmhJanBiSW1ScFpHTnZiVzB2ZGpJaVhTd2ljaUk2Vz

### Bobs cread DID and msg routed via Mediator

In [57]:
bob_did_to_mediator = await create_peer_did(1,1, service_endpoint="https://www.example.com/bob")
print(bob_did_to_mediator)

did:peer:2.Ez6LSmPiqTFZqFH9uoUEvZqjWMwhrzhTUxaFfruo5Cqk8NmAJ.Vz6MkgWpJvjRVD1JzZi5r2kzyhahLwVsFUYXcMQ6zGweGj3p8.SeyJpZCI6IiNkaWRjb21tIiwidCI6ImRtIiwicyI6eyJhIjpbImRpZGNvbW0vdjIiXSwiciI6W10sInVyaSI6Imh0dHA6Ly9sb2NhbGhvc3Q6ODA4MCJ9fQ


In [58]:
bob_routed_message = Message(
    id = str(uuid.uuid4()),
    type="https://didcomm.org/routing/2.0/forward",
    body={"next": alice_did},
    to=[mediator_did],
    attachments=[Attachment(
            data=AttachmentDataJson(json=json.loads(bob_basic_message_packed.packed_msg))
        )]
)

In [60]:
bob_routed_message_packed = await pack_encrypted(
    resolvers_config = ResolversConfig(
        secrets_resolver = secrets_resolver,
        did_resolver = DIDResolverPeerDID()
    ),
    message = bob_routed_message,
    frm = bob_did_to_mediator,
    to = mediator_did,
    sign_frm = None,
    pack_config = PackEncryptedConfig(protect_sender_id=False)
)

DIDCommValueError: `to` value is not a valid DID of DID URL: http://localhost:8080

### Sending to Mediator

In [ ]:
headers = {"Content-Type": "application/didcomm-encrypted+json"}
resp = requests.post(mediator_endpoint, headers=headers, json = json.loads(bob_routed_message_packed.packed_msg))


In [ ]:
print(resp)